# AI for policy and decisions {#sec-ai-decide}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/c21-ai-decide.ipynb)

The board of @sec-model-to-decision has EUR 20 million and one meeting. It wants the best use of the money, an honest account of what could go wrong, and a page it can read in two minutes. Language models can help with every part of that: an agent can search for a policy, a reasoning model can attack it, and a generative model can draft the brief. Each can also mislead, fluently. This chapter shows how to use all three on the waiting-list policy model, and how to check every output by code before a person signs it.

```
                levers                                scenarios
 +---------+  ----------->  +----------------------+  <----------  +-----------+
 |  Agent  |                | Chapter 15's model:  |               | Red team  |
 | (search)|  <-----------  | four bands, cost,    |  ---------->  | (reasoning|
 +---------+   measures     | budget               |   measures    |   model)  |
      |                     +----------------------+               +-----------+
      | check: numbers                  |                                |
      | from tools,                     v                                | check:
      | rerun, budget          +------------------+                      | ranges
      +----------------------> |  results table   | <--------------------+
                               +------------------+
                                        |
                                        v
                          +---------------------------+     check: numbers,
                          | Brief and board           | --> certainty, loops --> a person
                          | explanation (generative)  |                          edits, signs
                          +---------------------------+
```

*The chapter in one picture: models propose, search, attack and draft; the policy model computes every number; code checks each output before a person decides.*

The chapter keeps @sec-model-to-decision's model, levers and measures exactly, and adds a budget. Section 1 lets an agent search for a policy and checks its answer. Section 2 moves surrogate models here from machine learning and offers one to an agent as a fast tool. Section 3 asks a reasoning model to break the chosen policy. Sections 4 and 5 draft the brief and the board's explanation. A failure gallery closes the chapter: five ways AI goes wrong at this stage, each caught by a check.

**Who it is for.** Analysts who will use language models to support a policy recommendation, and anyone who must review such work. **Who can skip.** Readers who will not use AI at the decision stage can read the failure gallery alone; it shows what to look for in other people's work. **Prerequisites.** @sec-model-to-decision, all of it; @sec-ai-partners for language models, agents, replay files and the contract; @sec-story-to-diagram for the named feedback loops; @sec-calibration for calibration. **Time.** About three hours.

**Learning outcomes.** By the end of this chapter you can:

1. **build** a policy-search agent with a simulation tool and a budget, and **verify** its answer by code (section 1, Try it);
2. **validate** a surrogate model and **confirm** its answers with the real model (section 2, Try it);
3. **screen** scenarios proposed by a reasoning model and **judge** robustness with a regret table (section 3, Try it);
4. **check** the numbers and the certainty of a drafted brief, and **justify** each edit before signing (section 4, Try it);
5. **test** that an explanation names only feedback loops that exist in the diagram (section 5, Try it);
6. **diagnose** the ways AI fails at the decision stage, from the failure gallery (mini-project).

In [ ]:
# Run this cell first. In Colab it installs loopbricks (about ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
import inspect
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from loopbricks import ai

SEED = 21
rng = np.random.default_rng(SEED)

GITHUB = ("https://raw.githubusercontent.com/waelrash1/loopbricks/main/"
          "notebooks/")
DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") \
    else GITHUB + "data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])
# the recorded model replies of this chapter (see the note below)
REPLAY = "data/ai/c21.json" if os.path.exists("data/ai/c21.json") \
    else GITHUB + "data/ai/c21.json"

# Okabe-Ito colours, each with its own line style, so figures read in
# greyscale. rcParams holds matplotlib's defaults; cycler gives each new
# line the next colour and style together.
OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00",
             "#0072B2", "#CC79A7"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = cycler(color=OKABE_ITO) + \
    cycler(linestyle=STYLES)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| model for live calls:", ai.MODEL)

**The replies in this chapter.** Every model reply below is recorded in the replay file `data/ai/c21.json`, as @sec-ai-partners explains. We wrote them for the book with claude-opus-5-5; none came from a live API call, and each record says so in its `how` field (Check it prints them). We wrote them as a capable model would reply to the exact prompt and tool results shown. Where a reply contains faults on purpose, for teaching, the text says so.

To run the chapter against a live model, install the client (`pip install anthropic`), set `ANTHROPIC_API_KEY` to your key and set `LB_AI_LIVE=1`. Requests with no recorded reply then go to the model, and its replies are added to the replay file. A live model will search, attack and draft differently, and later prompts change with it. The checks do not change: they are the part of the chapter you keep.

## Chapter 15's model, rebuilt

The next cell rebuilds the policy model of @sec-model-to-decision without change: four bands of patients by months waited, treatments shared by priority weights, validation, and a stock of extra spending. Read it there. One new helper, `measures`, gives the chapter's four measures of a run, rounded as a report would give them.

In [ ]:
# Chapter 15's policy model, unchanged: four bands of patients by months
# waited, ageing between them, treatments shared by priority weights,
# validation, and a stock that adds up the extra spending.
last = wl.iloc[-1]                       # December 2023, where the runs start


def stock(i, name, init, units="patients"):
    return {"id": i, "name": name, "init": str(init), "units": units,
            "nonNegative": True}


def flow(i, name, eq, src=None, dst=None, units="patients/month"):
    f = {"id": i, "name": name, "eq": eq, "units": units}
    if src:
        f["from"] = src
    if dst:
        f["to"] = dst
    return f


def var(i, name, eq, units=""):
    return {"id": i, "name": name, "eq": str(eq), "units": units}


BANDS = ["Waiting 0 to 3 months", "Waiting 3 to 6 months",
         "Waiting 6 to 12 months", "Waiting over 12 months"]
B = [b.replace(" ", "_") for b in BANDS]          # names in equations
COLS = ["w0_3", "w3_6", "w6_12", "w12_plus"]
ON = "STEP(1, Policy_start)"                      # 1 once a policy starts

policy_spec = {
    "name": "Waiting list policy model", "timeUnit": "month",
    "T": 60, "dt": 0.25,
    "stocks": [stock(f"b{k}", BANDS[k - 1], int(last[COLS[k - 1]]))
               for k in range(1, 5)]
    + [stock("cost", "Extra cost", 0, "EUR million")],
    "flows": [
        flow("ref", "Referrals", "Referral_trend * (1 - Demand_reduction"
             " * MIN(1, MAX(0, TIME - Policy_start) / 12))", dst="b1"),
        flow("a1", "Ageing 1", f"{B[0]} / 3", "b1", "b2"),
        flow("a2", "Ageing 2", f"{B[1]} / 3", "b2", "b3"),
        flow("a3", "Ageing 3", f"{B[2]} / 6", "b3", "b4"),
    ]
    + [flow(f"t{k}", f"Treating {k}",
            f"Treatments * Weight_{k} * {B[k - 1]} / Weighted_list",
            f"b{k}") for k in range(1, 5)]
    + [flow(f"v{k}", f"Validating {k}", f"{B[k - 1]} * Validation_rate",
            f"b{k}") for k in range(1, 5)]
    + [flow("spend", "Spending",
            "(Extra_capacity * Cost_permanent + Initiative_capacity"
            " * Cost_initiative + Validation_contacts * Cost_contact)"
            f" / 1000000 + Demand_programme_cost * {ON}"
            " * (Demand_reduction > 0)", dst="cost",
            units="EUR million/month")],
    "vars": [
        var("wl", "Waiting list", " + ".join(B), "patients"),
        var("rb", "Referrals base", 5400, "patients/month"),
        var("rg", "Referral growth", 0.03, "fraction/year"),
        var("rt", "Referral trend",
            "Referrals_base * (1 + Referral_growth) ^ (TIME / 12)",
            "patients/month"),
        var("cb", "Capacity base", 5380, "patients/month"),
        var("cg", "Capacity growth", 0.02, "fraction/year"),
        var("ct", "Capacity trend",
            "Capacity_base * (1 + Capacity_growth) ^ (TIME / 12)",
            "patients/month"),
        var("ps", "Policy start", 3, "month"),
        var("pe", "Permanent extra", 0, "fraction"),
        var("te", "Temporary extra", 0, "fraction"),
        var("td", "Temporary duration", 12, "month"),
        var("ec", "Extra capacity",
            f"Capacity_trend * Permanent_extra * {ON}", "patients/month"),
        var("ic", "Initiative capacity",
            "Capacity_trend * Temporary_extra * (STEP(1, Policy_start)"
            " - STEP(1, Policy_start + Temporary_duration))",
            "patients/month"),
        var("cap", "Capacity",
            "Capacity_trend + Extra_capacity + Initiative_capacity",
            "patients/month"),
        var("mtt", "Minimum treatment time", 0.5, "month"),
        var("tr", "Treatments",
            "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
            "patients/month"),
        var("w1", "Weight 1", 1.4), var("w2", "Weight 2", 1.0),
        var("w3", "Weight 3", 0.9), var("w4", "Weight 4", 0.8),
        var("wsum", "Weighted list",
            " + ".join(f"Weight_{k} * {B[k - 1]}" for k in range(1, 5)),
            "patients"),
        var("dr", "Demand reduction", 0, "fraction"),
        var("dpc", "Demand programme cost", 0.3, "EUR million/month"),
        var("vf", "Validation fraction", 0, "fraction"),
        var("vd", "Validation duration", 6, "month"),
        var("vr", "Validation rate",
            "Validation_fraction / Validation_duration"
            " * (STEP(1, Policy_start)"
            " - STEP(1, Policy_start + Validation_duration))", "1/month"),
        var("vc", "Validation contacts",
            "Waiting_list / Validation_duration * (Validation_rate > 0)",
            "patients/month"),
        var("cpe", "Cost permanent", 1100, "EUR/patient"),
        var("cin", "Cost initiative", 1800, "EUR/patient"),
        var("cco", "Cost contact", 25, "EUR/patient"),
    ],
}
pm = lb.Model.from_spec(policy_spec)


def year(t):
    # model time in months from the end of 2023 -> calendar year
    return 2023 + 11 / 12 + np.asarray(t) / 12


POLICIES = {
    "Do nothing": {},
    "Permanent +5%": {"permanent_extra": 0.05},
    "Initiative +20%, 1 year": {"temporary_extra": 0.20},
    "Demand management": {"demand_reduction": 0.04},
    "Validate the list": {"validation_fraction": 0.10},
    "Longest waiters first": {"weight_4": 2.0},
    "Package": {"permanent_extra": 0.05, "validation_fraction": 0.10,
                "weight_4": 2.0},
}


def measures(d):
    # chapter 15's measures of one run, rounded as a report would give them
    dt = d.index[1] - d.index[0]
    over12 = d["Waiting over 12 months"]
    return {"long-wait months (k)": round(float(over12.sum() * dt) / 1000, 1),
            "over 12 m, end 2028": int(round(over12.iloc[-1])),
            "list, end 2028": int(round(d["Waiting list"].iloc[-1])),
            "cost (EUR m)": round(float(d["Extra cost"].iloc[-1]), 1)}


central = pd.DataFrame({n: measures(pm.run(**p))
                        for n, p in POLICIES.items()}).T
central = central.astype({"over 12 m, end 2028": int, "list, end 2028": int})
print(central.to_string())

The table reproduces @sec-model-to-decision. Doing nothing leaves 38,780 patients waiting over 12 months at the end of 2028. Over the five years it gives 2,153.8 thousand **long-wait months**: patients over 12 months, added up month by month. The package of three measures cuts these to 16,260 and 1,223.7 thousand, for EUR 19.4 million.

## 1. An agent that searches for a policy

**Stage task:** find the best combination of levers within a budget. **Kind of AI:** an agent (@sec-ai-partners), which chooses tool calls in a loop and reads their results [@yao2022react].

In @sec-model-to-decision we compared seven hand-picked policies. The board now asks a different question: which combination of levers is best for EUR 20 million? That is a search, and an agent can do one. It needs a tool, a Python function it may call. Ours takes the levers in the units people use, runs the model to the end of 2028 and returns the measures, the cost and whether the policy fits the budget.

Two design choices matter more than the prompt. First, the tool refuses lever values outside the ranges the board would consider, so the agent cannot wander off them. Second, validation is a switch, not a dial. How many patients a validation removes is something we do not control; it is an uncertain yield, about 10%, that section 3 treats as a scenario. The failure gallery shows what happens when a tool lets an agent treat it as a lever.

In [ ]:
BUDGET = 20.0       # EUR million over five years: the board's limit


def settings(permanent_pct=0.0, initiative_pct=0.0, demand_programme=False,
             validate=False, long_wait_weight=0.8):
    # the board's levers, in the units people use, as model constants
    return {"permanent_extra": permanent_pct / 100,
            "temporary_extra": initiative_pct / 100,
            "demand_reduction": 0.04 if demand_programme else 0.0,
            "validation_fraction": 0.10 if validate else 0.0,
            "weight_4": long_wait_weight}


def simulate_policy(permanent_pct=0.0, initiative_pct=0.0,
                    demand_programme=False, validate=False,
                    long_wait_weight=0.8):
    """Run the waiting-list model to the end of 2028, levers from month 3.

    permanent_pct: extra permanent capacity, percent, 0 to 10.
    initiative_pct: extra capacity for 12 months only, percent, 0 to 30.
    demand_programme: True runs demand management (referrals 4% lower).
    validate: True contacts everyone on the list over 6 months.
    long_wait_weight: priority weight of the over-12 band, 0.8 to 2.0.
    Returns the measures, the cost and whether it fits the EUR 20 m budget.
    """
    for name, v, lo, hi in (("permanent_pct", permanent_pct, 0, 10),
                            ("initiative_pct", initiative_pct, 0, 30),
                            ("long_wait_weight", long_wait_weight, 0.8, 2.0)):
        if not lo <= v <= hi:
            raise ValueError(f"{name} must be between {lo} and {hi}")
    d = pm.run(**settings(permanent_pct, initiative_pct, demand_programme,
                          validate, long_wait_weight))
    out = measures(d)
    out["within budget"] = bool(d["Extra cost"].iloc[-1] <= BUDGET)
    out["budget left (EUR m)"] = round(BUDGET - out["cost (EUR m)"], 1)
    return out


print(simulate_policy(permanent_pct=5, validate=True, long_wait_weight=2.0))

The agent receives the task, a short system text, and the tool's signature and docstring. It replies with one JSON object per step, as `ai.PROTOCOL` asks: a thought and a tool call, or a final answer. The tool runs here, in Python, and its result goes back as the next message. We ask for the settings on a line of their own, so code can read them.

In [ ]:
SYSTEM = ("You are a careful policy analyst working with a system dynamics "
          "model of a hospital waiting list. Test ideas with the tool; "
          "report only numbers the tool returned.")
TASK = ("The board can spend at most EUR 20 million over five years on its "
        "outpatient waiting list. Find the levers that give the fewest "
        "long-wait months (patients waiting over 12 months, added up month "
        "by month from 2024 to 2028) within that budget. Use simulate_policy "
        "for every number. Finish with one line 'settings: permanent_pct=..., "
        "initiative_pct=..., demand_programme=..., validate=..., "
        "long_wait_weight=...' and the measures the tool returned for it.")
steps = ai.agent(TASK, {"simulate_policy": simulate_policy}, system=SYSTEM,
                 replay=REPLAY, max_steps=12)
ai.show(steps, width=150)

The agent took nine runs. It used the free lever first (the long-wait weight), then the cheap one (validation), then compared three ways to spend the rest: permanent capacity, an initiative and demand management. It tried a mix, overshot the budget at a 16% initiative, and settled on 15.5%. Its answer reports a trade-off it was not asked about: the permanent package ends 2028 with fewer long waiters (16,260 against 19,704) and a shorter list.

How good is a nine-run search? The plain alternative is a grid: every combination of the levers in sensible steps, all in one `run_batch` call.

In [ ]:
# The plain alternative: every combination on a grid, all in one batch.
levels = {"permanent_pct": range(0, 11), "initiative_pct": range(0, 31, 5),
          "demand_programme": (False, True), "validate": (False, True),
          "long_wait_weight": (0.8, 1.1, 1.4, 1.7, 2.0)}
combos = pd.MultiIndex.from_product(levels.values(), names=levels.keys())
combos = combos.to_frame(index=False)     # one row per combination
batch = pm.run_batch(pd.DataFrame([settings(**row) for row in
                                   combos.to_dict("records")]),
                     include=["Waiting over 12 months", "Extra cost"])
over12 = batch[batch.variable == "Waiting over 12 months"]
combos["long-wait months (k)"] = (over12.groupby("run")["value"].sum()
                                  * 0.25 / 1000).to_numpy()
combos["cost (EUR m)"] = batch[(batch.variable == "Extra cost")
                               & (batch.time == 60)]["value"].to_numpy()
fits = combos[combos["cost (EUR m)"] <= BUDGET]
best_grid = fits.sort_values("long-wait months (k)").iloc[0]
print(f"{len(combos)} grid runs, {len(fits)} within budget. Best:")
print(best_grid.round(2).to_string())
print("Agent's tool calls:", sum("tool" in s for s in steps))

The grid has 1,540 runs, of which 145 fit the budget. Its best is a 15% initiative with validation and a weight of 2.0: 1,188.8 thousand long-wait months for EUR 19.3 million. The agent reached the same region in nine runs and went a step further, to 15.5%, between two grid points.

Neither result is trusted yet. The contract of @sec-ai-partners says the numbers come from tools, and that a check enforces it. The cell below runs four checks on the answer. Every number in it must come from a tool or the task. The recommended settings must have been run in the search, and must rerun to the same measures. And the policy must fit the budget.

In [ ]:
def numbers(text):
    # every number written in a text, as floats: "1,188.8" -> 1188.8;
    # (?<![\w.]) skips digits inside names such as B1
    return [float(x.replace(",", "")) for x in
            re.findall(r"(?<![\w.])\d[\d,]*(?:\.\d+)?", text)]


def unsupported(text, sources):
    # the numbers in text that appear in none of the sources
    seen = {round(v, 6) for s in sources for v in numbers(str(s))}
    return [v for v in numbers(text) if round(v, 6) not in seen]


def parse_settings(answer):
    # the 'settings: a=1, b=True, ...' line of an answer, as a dict
    line = re.search(r"settings:(.*)", answer).group(1)
    pairs = dict(re.findall(r"(\w+)=([\w.]+)", line))
    return {k: (v == "True") if v in ("True", "False") else float(v)
            for k, v in pairs.items()}


answer = steps[-1]["answer"]
calls = [s for s in steps if "tool" in s]
sources = [TASK] + [s["args"] for s in calls] + [s["result"] for s in calls]
pick = parse_settings(answer)
# settings() fills in the levers an agent left at their defaults
same = [s["result"] for s in calls
        if settings(**s["args"]) == settings(**pick)]
rerun = simulate_policy(**pick)
print("Numbers in the answer that no tool returned:",
      unsupported(answer, sources))
print("Recommended settings:", pick)
print("The tool returned them in the search:", len(same) > 0)
print("Rerun gives the same measures:", len(same) > 0 and rerun == same[0])
print("Within the budget:", rerun["within budget"])
print("Agent's pick against the best grid point (long-wait months, k):",
      rerun["long-wait months (k)"], "against",
      round(best_grid["long-wait months (k)"], 1), "| difference",
      round(best_grid["long-wait months (k)"]
            - rerun["long-wait months (k)"], 1))

All four checks pass. `unsupported` finds every number in the answer and looks for it in the task, the tool arguments and the tool results. An empty list means the agent quoted only what the tools returned. The rerun matters because an answer can quote real numbers from the wrong run. The agent's 1,178.9 thousand beats the grid's 1,188.8 thousand, by 9.9 thousand long-wait months.

In [ ]:
#| fig-alt: Scatter of 1,540 grid policies by extra cost and long-wait months, with the agent's nine runs joined in order; the best policies sit just left of the EUR 20 million budget line.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(combos["cost (EUR m)"], combos["long-wait months (k)"], s=6,
           color="#BBBBBB", label=f"grid, {len(combos)} runs")
tried = pd.DataFrame([s["result"] for s in calls
                      if isinstance(s["result"], dict)])
ax.plot(tried["cost (EUR m)"], tried["long-wait months (k)"], "o-",
        color=OKABE_ITO[4], ms=6, label=f"agent, {len(tried)} runs in order")
ax.axvline(BUDGET, color="black", lw=1, ls="--", label="budget")
ax.set(title="Policy search: a grid and an agent",
       xlabel="extra cost over five years (EUR million)",
       ylabel="long-wait months (thousand)", xlim=(-2, 60))
ax.legend()
plt.show()

*Figure: every grid run (grey) and the agent's nine runs in order (joined), by cost and long-wait months. The dashed line is the EUR 20 million budget. The best policies sit just left of it.*

The figure shows why the agent did well: the best policies lie along the budget line, and a model that reads costs can head straight there. A grid wastes most of its runs far from that line. With five levers a grid is still cheap here; with twenty levers, or a model that takes minutes, it is not.

**What stays human.** The agent optimised what we asked for, long-wait months over five years. It found that an initiative wins on that measure, and noted that permanent capacity wins on where the list ends. Which matters more is the board's judgement, not the agent's. The agent's pick becomes Plan A, and the package becomes Plan B; section 3 tests both.

**Try it 1.**

- *Check.* How many runs did the agent use, and how many did the grid use? Which found the better policy within budget?
- *Apply.* A clinical lead says a long-wait weight above 1.4 would delay urgent patients too much. Find the best grid point with the weight at most 1.4. Does the best use of the money change?
- *Extend.* The board cuts the budget to EUR 15 million. Find the best grid point. Would you expect the agent to find the same? Write the task you would give it.

<details><summary>Worked answer</summary>

*Check.* Nine runs for the agent, 1,540 for the grid. The agent's 15.5% initiative (1,178.9 thousand long-wait months) beats the grid's best, a 15% initiative (1,188.8 thousand), because 15.5% lies between grid points.

*Apply.* The money is spent the same way: a 15% initiative with validation, now with the weight at 1.4. Long-wait months rise to 1,327.9 thousand. The weight is a free lever, so capping it costs results but does not change how the money is best spent.

*Extend.* Within EUR 15 million the best grid point mixes 2% permanent capacity with a 5% initiative and validation, at a weight of 2.0. It gives 1,283.3 thousand long-wait months for EUR 14.6 million. A new budget is a new prompt, so the replay file has no reply for it; run it live to see. Whatever the agent finds, the four checks of this section apply unchanged.

</details>

In [ ]:
print("Agent runs:", len(tried), "| grid runs:", len(combos))
capped = fits[fits["long_wait_weight"] <= 1.4]
print("Best with the weight at most 1.4:")
print(capped.sort_values("long-wait months (k)").iloc[0].round(2)
      .to_string())
tight = combos[combos["cost (EUR m)"] <= 15]
print("Best within EUR 15 million:")
print(tight.sort_values("long-wait months (k)").iloc[0].round(2).to_string())

## 2. Machine learning: surrogate models

**Stage task:** search fast when each run of the model is slow. **Kind of AI:** machine learning (a statistical model fitted to runs), then an agent that uses it as a tool.

Policy search, calibration and uncertainty analysis need thousands of runs. When one run takes minutes, as in a large or agent-based model, a **surrogate** helps. Also called a **metamodel** or **emulator**, it is a cheap statistical model fitted to a few hundred runs of the real one. It does the searching, and the real model checks its answer.

We use the one-stock case model of the forecasting contest in @sec-ai-test, calibrated to the data up to 2021 as in @sec-calibration, and run on to the end of 2028. Two levers start in January 2024: `Extra capacity` (a fraction, 0 to 0.15) and `Demand cut` (a fraction, 0 to 0.10). The outcome is the list at the end of 2028. The next cell rebuilds the model and its calibration.

In [ ]:
# Chapter 13's one-stock case model, calibrated to the data up to 2021 as in
# chapter 14's forecasting contest, with two levers from January 2024.
y = wl["total"].to_numpy(dtype=float)
TRAIN_END = 96                                   # Jan 2022 is month 96

case = lb.Model.from_spec({
    "name": "Waiting list case", "timeUnit": "month", "T": 119, "dt": 0.25,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": str(y[0]),
                "units": "patients", "nonNegative": True}],
    "flows": [
        {"id": "r", "name": "Referrals", "to": "wl",
         "eq": "Referrals_base * (1 + Referral_growth) ^ (TIME / 12)"
               " * (1 - Demand_cut * STEP(1, Lever_start))",
         "units": "patients/month"},
        {"id": "t", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"}],
    "vars": [
        {"id": "rb", "name": "Referrals base", "eq": "4000",
         "units": "patients/month"},
        {"id": "rg", "name": "Referral growth", "eq": "0.03",
         "units": "fraction/year"},
        {"id": "cb", "name": "Capacity base", "eq": "3950",
         "units": "patients/month"},
        {"id": "cg", "name": "Capacity growth", "eq": "0.02",
         "units": "fraction/year"},
        {"id": "pan", "name": "Pandemic effect",
         "eq": "TABLE(TIME, 73.9, 1, 74, 0.35, 83, 1)", "units": "1"},
        {"id": "ini", "name": "Initiative size", "eq": "0.10",
         "units": "fraction"},
        {"id": "ls", "name": "Lever start", "eq": "120", "units": "month"},
        {"id": "xc", "name": "Extra capacity", "eq": "0",
         "units": "fraction"},
        {"id": "dc", "name": "Demand cut", "eq": "0", "units": "fraction"},
        {"id": "cap", "name": "Capacity",
         "eq": "Capacity_base * (1 + Capacity_growth) ^ (TIME / 12)"
               " * Pandemic_effect * (1 + Initiative_size * STEP(1, 96))"
               " * (1 + Extra_capacity * STEP(1, Lever_start))",
         "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"}]})
KEYS = ["referrals_base", "referral_growth", "capacity_base",
        "capacity_growth"]


def simulate(theta, T=119, **kw):
    d = case.run(T=T, include=["Waiting list"], **dict(zip(KEYS, theta)),
                 **kw)
    return d.loc[np.arange(T + 1.0), "Waiting list"].to_numpy()


# A short Nelder-Mead with lists of points: np.argsort sorts them best first,
# and 2 * c - worst reflects the worst point through the centre c of the rest
def nelder_mead(f, x0, step, iters=500):
    pts = [np.array(x0, float)] + [np.array(x0, float) + np.eye(len(x0))[i]
                                   * step[i] for i in range(len(x0))]
    val = [f(p) for p in pts]
    for _ in range(iters):
        order = np.argsort(val)
        pts, val = [pts[i] for i in order], [val[i] for i in order]
        c = np.mean(pts[:-1], axis=0)
        xr = 2 * c - pts[-1]
        fr = f(xr)
        if fr < val[0]:
            xe = 3 * c - 2 * pts[-1]
            fe = f(xe)
            pts[-1], val[-1] = (xe, fe) if fe < fr else (xr, fr)
        elif fr < val[-2]:
            pts[-1], val[-1] = xr, fr
        else:
            xc = (c + pts[-1]) / 2
            fc = f(xc)
            if fc < val[-1]:
                pts[-1], val[-1] = xc, fc
            else:
                pts = [pts[0]] + [(p + pts[0]) / 2 for p in pts[1:]]
                val = [val[0]] + [f(p) for p in pts[1:]]
    return pts[int(np.argmin(val))]


def calibrate(end):
    sse = lambda th: ((simulate(th, T=end - 1) - y[:end]) ** 2).sum()
    return nelder_mead(sse, [3800, 0.02, 3800, 0.01],
                       [200, 0.01, 200, 0.01])


theta = calibrate(TRAIN_END)
calibrated = dict(zip(KEYS, theta))
print({k: round(float(v), 4) for k, v in calibrated.items()})

The fit gives referrals of 3,553 patients/month growing 6.4% a year and capacity of 3,508 patients/month growing 5.4% a year. The stock pins down mostly the difference between the two, as @sec-calibration discusses, and the difference is what drives the list.

We fit a quadratic surrogate to 40 runs and test it on 400 others, all with `run_batch`.

In [ ]:
T_END = 179                                       # December 2028


def outcome(levers):
    # .assign adds the calibrated constants as columns, the same in every row
    settings_ = pd.DataFrame(levers).assign(**calibrated)
    out = case.run_batch(settings_, T=T_END, include=["Waiting list"])
    return out[out.time == T_END].sort_values("run")["value"].to_numpy()


def design(n, seed):
    g = np.random.default_rng(seed)
    return {"extra_capacity": g.uniform(0, 0.15, n),
            "demand_cut": g.uniform(0, 0.10, n)}


def features(lev):
    a, b = np.asarray(lev["extra_capacity"]), np.asarray(lev["demand_cut"])
    # np.ones_like(a): ones in the shape of a, for the constant term
    return np.column_stack([np.ones_like(a), a, b, a * a, a * b, b * b])


fit_runs, test_runs = design(40, SEED), design(400, SEED + 1)
y_fit, y_test = outcome(fit_runs), outcome(test_runs)
beta = np.linalg.lstsq(features(fit_runs), y_fit, rcond=None)[0]
pred = features(test_runs) @ beta          # @: each row times the coefficients
r2 = 1 - ((pred - y_test) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()
print(f"Surrogate on 400 unseen runs: R^2 = {r2:.4f}, largest error "
      f"{np.abs(pred - y_test).max():.0f} patients")

The quadratic surrogate explains 98.7% of the variation in the 400 unseen runs ($R^2$ = 0.9869), but its largest error is 9,194 patients: good on average, poor in places.

Now search. Suppose each percentage point of extra capacity costs EUR 3.5 million over the five years, and each point of demand cut EUR 2 million (illustrative prices). Which is the cheapest pair that brings the list below 50,000 by the end of 2028? The surrogate scores a fine grid of 10,201 pairs at once, and the real model checks its pick.

In [ ]:
# np.meshgrid gives every pair of the two lists as 2-D arrays; .ravel()
# flattens each into one long list of candidates
xa, xb = np.meshgrid(np.linspace(0, 0.15, 101), np.linspace(0, 0.10, 101))
cand = {"extra_capacity": xa.ravel(), "demand_cut": xb.ravel()}
guess = features(cand) @ beta
cost = 350 * cand["extra_capacity"] + 200 * cand["demand_cut"]  # EUR m
ok = guess < 50000
i = int(np.argmin(np.where(ok, cost, np.inf)))   # cheapest of those that pass
first = {k: v[i:i + 1] for k, v in cand.items()}  # [i:i + 1]: a one-item array
real = outcome(first)[0]
print(f"Surrogate pick: extra capacity {first['extra_capacity'][0]:.1%}, "
      f"demand cut {first['demand_cut'][0]:.1%}, cost EUR {cost[i]:.1f} m")
print(f"Surrogate says {guess[i]:.0f}; the real model says {real:.0f}")

The surrogate picks no extra capacity and a 7.2% cut in demand, for EUR 14.4 million. It predicts 49,766 patients; the real model says 49,528, so the pick meets the target. But is it the cheapest? An optimiser drives towards the edge of what the surrogate thinks is allowed, where a small error decides the answer. A safe method shortlists the candidates the surrogate likes, with a margin, and lets the real model decide among them.

In [ ]:
# np.flatnonzero lists the positions where the mask is True
near = np.flatnonzero(guess < 50000 + 3000)        # a safety margin
short = near[np.argsort(cost[near])][:300]         # the 300 cheapest
checked = outcome({k: v[short] for k, v in cand.items()})
j = short[np.flatnonzero(checked < 50000)[0]]
print(f"Checked pick: extra capacity {cand['extra_capacity'][j]:.1%}, "
      f"demand cut {cand['demand_cut'][j]:.1%}, cost EUR {cost[j]:.1f} m, "
      f"real model {checked[short == j][0]:.0f}, after {len(short)} runs")

The real model finds a cheaper pair: a 7.1% cut, for EUR 14.2 million, at 49,982 patients. The surrogate had put it just above the target, so it never offered it. Here the surrogate's error cost EUR 0.2 million; with another random design it can miss the target altogether, as the failure gallery shows. The shortlist method needed 300 real runs to be sure.

**The surrogate as an agent's tool.** An agent can do the shortlist more economically, if the rules are clear. We give it two tools: `surrogate_cheapest`, fast but approximate, and `real_model`, which we describe as slow. The task says any recommended pair must have been run on the real model.

In [ ]:
REAL_RUNS = []                       # every run of the real model, for checks


def surrogate_cheapest(target=50000, margin=0, n=5):
    """The n cheapest lever pairs the fast surrogate predicts are below
    target + margin patients at the end of 2028, from 10,201 pairs scored.

    Gives extra capacity and demand cut (percent), the surrogate's
    predicted list (patients) and the cost (EUR million). An estimate only.
    """
    keep = np.flatnonzero(guess < target + margin)
    best = keep[np.argsort(cost[keep], kind="stable")][:int(n)]
    return [{"extra_capacity_pct":
                 round(float(100 * cand["extra_capacity"][k]), 2),
             "demand_cut_pct": round(float(100 * cand["demand_cut"][k]), 2),
             "predicted list": int(round(guess[k])),
             "cost (EUR m)": round(float(cost[k]), 1)} for k in best]


def real_model(extra_capacity_pct, demand_cut_pct):
    """Run the calibrated model itself for one lever pair (slow in a large
    model, so use it sparingly). Gives the list at the end of 2028
    (patients) and the cost (EUR million)."""
    lev = {"extra_capacity": np.array([extra_capacity_pct / 100]),
           "demand_cut": np.array([demand_cut_pct / 100])}
    out = {"extra_capacity_pct": extra_capacity_pct,
           "demand_cut_pct": demand_cut_pct,
           "real list": int(round(outcome(lev)[0])),
           "cost (EUR m)": round(3.5 * extra_capacity_pct
                                 + 2 * demand_cut_pct, 1)}
    REAL_RUNS.append(out)
    return out


TASK2 = ("Find the cheapest pair of levers, extra capacity and a cut in "
         "demand, both from January 2024, that brings the waiting list below "
         "50,000 patients by the end of 2028. Each percentage point costs "
         "EUR 3.5 million for capacity and EUR 2 million for demand. The "
         "surrogate is fast but approximate; any pair you recommend must "
         "have been run on the real model. Finish with the pair, its real "
         "list and its cost.")
steps2 = ai.agent(TASK2, {"surrogate_cheapest": surrogate_cheapest,
                          "real_model": real_model},
                  system=SYSTEM, replay=REPLAY, max_steps=10)
ai.show(steps2, width=150)

The agent used the surrogate to find candidates and the real model to decide. When its first real run came in below the surrogate's estimate, it reasoned that cheaper pairs might pass too, asked for candidates with a margin and checked them cheapest first. It stopped at the first pass. The checks below confirm the answer by code.

In [ ]:
answer2 = steps2[-1]["answer"]
calls2 = [s for s in steps2 if "tool" in s]
sources2 = [TASK2] + [s["args"] for s in calls2] + [s["result"]
                                                    for s in calls2]
print("Numbers in the answer that no tool returned:",
      unsupported(answer2, sources2))
passed = [r for r in REAL_RUNS if r["real list"] < 50000]
final = min(passed, key=lambda r: r["cost (EUR m)"])
print("Cheapest pair the real model confirmed:", final)
again = real_model(final["extra_capacity_pct"], final["demand_cut_pct"])
print("Rerun agrees:", again["real list"] == final["real list"])
print("The answer quotes the real model's list:",
      f"{final['real list']:,}" in answer2)
print("Real-model runs by the agent:", len(REAL_RUNS) - 1,
      "| by the shortlist method:", len(short))

The agent reached the shortlist method's answer, a 7.1% cut for EUR 14.2 million at 49,982 patients, with four real runs instead of 300. Every number in its answer came from a tool, its pair reruns to the same list, and the answer quotes the real model's list, not the surrogate's.

With a model this small, a surrogate is not needed: `run_batch` scores all 10,201 pairs directly in seconds. The method earns its place when each run is slow. Three rules keep it safe. Test the surrogate on runs it was not fitted to. Confirm every answer with the real model, because a surrogate is least accurate where an optimiser likes to go. And check, by code, that an agent did so.

**What stays human.** The prices. Under these illustrative prices demand management is the cheaper lever; with other prices the answer changes. A surrogate makes such questions quick to explore, but the prices, and whether a 7% cut in demand is achievable at all, are judgements for people who know the service.

**Try it 2.**

- *Check.* At the agent's confirmed pair, what does the surrogate predict? Would the surrogate alone have accepted it?
- *Apply.* Fit a linear surrogate (drop the three squared and product terms). How do its $R^2$ and largest error on the 400 test runs compare?
- *Extend.* Check the surrogate at the edge of its range: what does it predict, and the model give, for 15% extra capacity and a 10% demand cut together?

<details><summary>Worked answer</summary>

*Check.* The surrogate predicts 50,247 patients at a 7.1% cut, above the target, so on its own it would have rejected the cheapest pair that works. The real model gives 49,982.

*Apply.* The linear surrogate has $R^2$ = 0.964 and a largest error of 22,385 patients, against 0.9869 and 9,194 for the quadratic one. The curvature matters.

*Extend.* The surrogate predicts -5,864 patients, an impossible negative list; the real model gives 4,011, held up by the limit `Waiting_list / Minimum_treatment_time`. The polynomial knows nothing about that floor.

</details>

In [ ]:
lin = np.linalg.lstsq(features(fit_runs)[:, :3], y_fit, rcond=None)[0]
pl = features(test_runs)[:, :3] @ lin
r2l = 1 - ((pl - y_test) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()
print(f"Linear surrogate: R^2 = {r2l:.4f}, largest error "
      f"{np.abs(pl - y_test).max():.0f}")
corner = {"extra_capacity": np.array([0.15]), "demand_cut": np.array([0.10])}
print(f"Corner: surrogate {(features(corner) @ beta)[0]:.0f}, "
      f"model {outcome(corner)[0]:.0f}")
k_pick = {"extra_capacity": np.array([final["extra_capacity_pct"] / 100]),
          "demand_cut": np.array([final["demand_cut_pct"] / 100])}
print(f"At the agent's pick the surrogate says "
      f"{(features(k_pick) @ beta)[0]:.0f}, "
      f"the model {outcome(k_pick)[0]:.0f}")

## 3. Reasoning: a red team for the chosen policy

**Stage task:** find the conditions under which a policy fails. **Kind of AI:** a reasoning model, which thinks before it answers (`reasoning=True`).

A **red team** is a group whose job is to break a plan before the world does. A reasoning model makes a tireless one: it can read the policy and the model's assumptions, and propose scenarios in which the policy does badly. The danger is that it proposes scenarios that are not plausible, or that the model cannot represent. So we state the ranges first, in code, and let code screen every proposal.

The ranges below are our working judgements, wider than the grid of @sec-model-to-decision. They are what a modelling team would agree with the service before the red team starts.

In [ ]:
PLAN_A = pick                                     # the agent's recommendation
PLAN_B = {"permanent_pct": 5, "validate": True, "long_wait_weight": 2.0}
PLANS = {"Do nothing": {}, "Plan A (agent)": PLAN_A,
         "Plan B (package)": PLAN_B}
# what we do not know, and how far it could plausibly go
PLAUSIBLE = {
    "referral_growth": (0.01, 0.05, "fraction/year"),
    "capacity_growth": (0.01, 0.03, "fraction/year"),
    "referrals_base": (5300, 5600, "patients/month"),
    "policy_start": (3, 9, "month"),
    "validation_yield": (0.05, 0.15, "fraction of the list removed"),
    "cost_permanent": (1000, 1250, "EUR/patient"),
    "cost_initiative": (1600, 2400, "EUR/patient"),
}
ranges = "\n".join(f"- {k}: {lo} to {hi} ({u})"
                   for k, (lo, hi, u) in PLAUSIBLE.items())
RED = f"""You are red-teaming a policy before it goes to a hospital board.
The model: chapter 15's waiting-list model (four bands by months waited,
treatments shared by priority weights), run from January 2024 to the end of
2028, with central values referral_growth 0.03, capacity_growth 0.02,
referrals_base 5400, policy_start 3, validation_yield 0.10,
cost_permanent 1100 and cost_initiative 1800.
The policy (Plan A): {PLAN_A}
Its central-case measures: {simulate_policy(**PLAN_A)}
The budget is EUR {BUDGET:.0f} million.

Propose up to six scenarios in which Plan A does badly: more long waits,
long waits rising again by 2028, or a cost over the budget. Each scenario
may change only these parameters, within these ranges:
{ranges}
Return only JSON: a list of objects with "name", "why" (one sentence on
the mechanism) and "changes" (parameter: value)."""
print(RED)

The prompt gives the red team Plan A, the agent's recommendation, with its central-case measures, and asks for scenarios as JSON. `ask_json` reads the first JSON list in the reply.

In [ ]:
proposed = ai.ask_json(RED, reasoning=True, replay=REPLAY)
for s in proposed:
    print(f"- {s['name']}: {s['changes']}")

The red team proposed six scenarios, each with a mechanism. They are premium rates for bought-in activity, a late start, demand outgrowing capacity, a weak validation with busier referrals, an ageing population and burnout after the blitz. The last two are the interesting ones, and code decides what happens to them.

In [ ]:
def screen(scenario):
    # reasons to reject a scenario, by code: unknown names, values out of range
    bad = []
    for k, v in scenario["changes"].items():
        if k not in PLAUSIBLE:
            bad.append(f"{k} is not a parameter of the model")
        elif not PLAUSIBLE[k][0] <= v <= PLAUSIBLE[k][1]:
            bad.append(f"{k} = {v} is outside {PLAUSIBLE[k][:2]}")
    return bad


accepted = [s for s in proposed if not screen(s)]
for s in proposed:
    print(f"{s['name']}: {'; '.join(screen(s)) or 'accepted'}")
# a person adds the mirror attack the red team was not asked for
accepted.append({"name": "Permanent costs overrun",
                 "why": "added by the modeller: the same attack on Plan B",
                 "changes": {"cost_permanent": 1250}})

Code rejects two of the six. A referral growth of 7% a year is outside the range we agreed; it may be worth discussing, but not slipped in as plausible. `staff_attrition` is not a parameter of the model: burnout is loop R2 of @sec-story-to-diagram, which the policy model leaves out. That rejection is information. It marks a boundary question for people, not a scenario to run.

The red team attacked only Plan A. A fair test attacks both, so the modeller adds the mirror attack: permanent treatments cost EUR 1,250 instead of 1,100. Now code runs every accepted scenario for doing nothing, Plan A and Plan B.

In [ ]:
def run_plan(plan, changes):
    # one plan in one scenario: the levers, then the scenario's changes
    p = settings(**plan)
    for k, v in changes.items():
        if k == "validation_yield":
            if p["validation_fraction"] > 0:     # only a plan that validates
                p["validation_fraction"] = v
        else:
            p[k] = v
    return measures(pm.run(**p))


scen = [{"name": "Central case", "changes": {}}] + accepted
rows = []
for s in scen:
    for plan, levers in PLANS.items():
        m = run_plan(levers, s["changes"])
        rows.append({"scenario": s["name"], "plan": plan, **m})
rt = pd.DataFrame(rows)
over = rt.pivot(index="scenario", columns="plan",
                values="over 12 m, end 2028").loc[[s["name"] for s in scen]]
lwm = rt.pivot(index="scenario", columns="plan",
               values="long-wait months (k)").loc[over.index]
spend = rt.pivot(index="scenario", columns="plan",
                 values="cost (EUR m)").loc[over.index]
print("Over 12 months at the end of 2028:")
print(over.to_string())
print("\nLong-wait months (thousand):")
print(lwm.to_string())
print("\nCost (EUR million):")
print(spend.to_string())

The tables answer the board's question, how sure are we, scenario by scenario. Plan B ends 2028 with fewer patients over 12 months than Plan A in all six, from 16,260 in the central case to 30,592 when demand outgrows capacity. Plan A has fewer long-wait months in all six, because it acts sooner. And the budget is tight for both. Plan A goes over it when premium rates rise (EUR 26.0 million) and, narrowly, with a late start (EUR 20.1 million). Plan B goes over when permanent costs overrun (EUR 21.8 million).

A **regret** is how much worse a plan does in a scenario than the best plan would have done in that same scenario (@sec-model-to-decision). The regret table compares the two plans.

In [ ]:
two = ["Plan A (agent)", "Plan B (package)"]
regret = pd.DataFrame({
    "worst over 12 m": over[two].max(),
    "max regret, over 12 m": (over[two].sub(over[two].min(axis=1),
                                            axis=0)).max(),
    "max regret, long-wait months (k)": (lwm[two].sub(lwm[two].min(axis=1),
                                                       axis=0)).max(),
    "scenarios over budget": (spend[two] > BUDGET).sum(),
    "of": len(scen)})
print(regret.round(1).to_string())
gap = (lwm["Plan B (package)"] - lwm["Plan A (agent)"]).round(1)
print("Plan B's extra long-wait months over Plan A (k):", gap.min(), "to",
      gap.max())

On patients over 12 months at the end of 2028, Plan B has no regret at all. Plan A's worst regret is 3,864 patients, with a weak validation and busier referrals. On long-wait months it is the other way round: Plan B's worst regret is 62.0 thousand, with a late start. Plan A goes over budget in two of the six scenarios, Plan B in one.

In [ ]:
#| fig-alt: Paired bars of patients waiting over 12 months at the end of 2028 for Plan A and Plan B in six scenarios; Plan B is lower in each, and both are highest when demand outgrows capacity.
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(over))
for k, plan in enumerate(two):
    ax.bar(x + (k - 0.5) * 0.38, over[plan] / 1000, width=0.38, label=plan,
           color=OKABE_ITO[1 + 4 * k], hatch=["", "//"][k], edgecolor="black")
ax.set_xticks(x, [n.replace(" ", "\n", 2) for n in over.index], fontsize=8)
ax.set(title="Red-team scenarios: patients waiting over 12 months, end 2028",
       ylabel="patients (thousand)")
ax.legend()
plt.tight_layout()
plt.show()

*Figure: patients waiting over 12 months at the end of 2028 under each plan, in the central case and the five red-team scenarios (the last added by the modeller). Plan B (hatched) is lower in every scenario.*

**The decision.** The board values where the list is heading and a budget it can keep. Plan B ends lower in every scenario and breaks the budget in one, under a cost overrun the board can watch for; Plan A breaks it in two. The board therefore chooses Plan B, accepting 44.6 to 62.0 thousand more long-wait months over five years, depending on the scenario. That is a judgement about values, made by people, with the trade-off in front of them.

**What stays human.** The ranges, the mirror attack, and the choice. The reasoning model was useful because it thought of mechanisms (premium rates, a late start) and said why; code made sure each one was plausible and ran it. Nothing it proposed became a number until the model computed it.

**Try it 3.**

- *Check.* Which two proposed scenarios did code reject, and for what reason?
- *Apply.* Run the rejected "Ageing population surge" anyway, outside the agreed range. Does Plan B still end 2028 lower than Plan A?
- *Extend.* Draw 200 random scenarios, each parameter uniform over its plausible range, and report how often Plan B ends 2028 lower than Plan A and how often each plan goes over budget.

<details><summary>Worked answer</summary>

*Check.* "Ageing population surge" (a referral growth of 0.07, outside 0.01 to 0.05) and "Burnout after the blitz" (`staff_attrition` is not a parameter of the model).

*Apply.* Yes: 34,638 patients over 12 months at the end of 2028 under Plan B against 38,737 under Plan A. The ranking holds even outside the range, but the numbers belong in a discussion, not in the brief.

*Extend.* Plan B ends lower than Plan A in all 200 draws. Plan A goes over budget in 72% of them and Plan B in 22%. When every cost may move at once, a plan that spends 19.4 of 20 million has little margin. That is worth a sentence to the board.

</details>

In [ ]:
out_of_range = [s for s in proposed if any(
    k == "referral_growth" and v > 0.05 for k, v in s["changes"].items())]
for s in out_of_range:
    print(s["name"], {p: run_plan(PLANS[p], s["changes"])
                      ["over 12 m, end 2028"] for p in two})
draws = []
g = np.random.default_rng(SEED)
for _ in range(200):
    ch = {k: g.uniform(lo, hi) for k, (lo, hi, _) in PLAUSIBLE.items()}
    a, b = (run_plan(PLANS[p], ch) for p in two)
    draws.append({"B better at end 2028":
                      b["over 12 m, end 2028"] < a["over 12 m, end 2028"],
                  "A over budget": a["cost (EUR m)"] > BUDGET,
                  "B over budget": b["cost (EUR m)"] > BUDGET})
print(pd.DataFrame(draws).mean().round(2).to_string())

## 4. Generative: drafting the brief

**Stage task:** write the one-page brief of @sec-model-to-decision for the chosen plan. **Kind of AI:** generative.

A drafting model writes fluent, well-ordered sentences quickly. It also rounds, paraphrases and, sometimes, computes or invents a number of its own [@ji2023hallucination]. So the model sees only a results table built by code, and code checks every number in its draft against that table. The table holds the central case for the three plans, today's list and the red-team results for Plan B.

In [ ]:
CHOSEN = "Plan B (package)"
facts = pd.DataFrame({p: run_plan(PLANS[p], {}) for p in PLANS}).T
facts = facts.astype({"over 12 m, end 2028": int, "list, end 2028": int})
breach = spend.index[spend[CHOSEN] > BUDGET]       # scenarios over budget
robust = {
    "today (December 2023): list": int(last["total"]),
    "today (December 2023): over 12 m": int(last["w12_plus"]),
    "scenarios tested, central case included": len(scen),
    "Plan B, over 12 m at end 2028, lowest": int(over[CHOSEN].min()),
    "Plan B, over 12 m at end 2028, highest": int(over[CHOSEN].max()),
    "scenarios where Plan B ends 2028 with fewer over 12 m than Plan A":
        int((over[CHOSEN] < over["Plan A (agent)"]).sum()),
    "scenarios where Plan B goes over budget": len(breach),
    "which": ", ".join(breach) + " (cost of a permanent treatment EUR "
             "1250 instead of 1100)",
    "budget (EUR m)": BUDGET}
TABLE = facts.to_string() + "\n\n" + "\n".join(f"{k}: {v}"
                                                for k, v in robust.items())
print(TABLE)

The prompt asks for five or six short sentences, the shape of the brief in @sec-model-to-decision, and for the numbers in the table only. This reply was written as a good draft typically reads, including two slips that drafts often contain. Can you spot them before the check does?

In [ ]:
BRIEF = f"""Draft a brief of five or six short sentences for a hospital board
that does not read code. The board chose Plan B: 5% more permanent capacity,
a check of who still needs care, and seeing the longest waiters first. Use
only the numbers in the tables below, rounded as a careful analyst would;
say what is happening, what Plan B buys, what it costs, and how sure we are.

{TABLE}"""
draft = ai.ask(BRIEF, replay=REPLAY)
print(draft)

Two checks run on the draft. `check_brief` finds every number and its nearest value in the prompt's table, and accepts it within 3%, the rounding a careful analyst would use. It also accepts thousands written as millions, which makes it slightly looser. `certainty` flags sentences with words that claim more than a model can give: "will", "guarantee", "always", "every" and the like. A flag is not a verdict; it tells a person where to look.

In [ ]:
CERTAIN = r"\b(will|guarantee\w*|certain\w*|definitely|ensure\w*|always|" \
          r"never|eliminat\w*|prove\w*|every)\b"


def check_brief(text, table, tol=0.03):
    # each number in text against the table: a match within tol, or a flag
    allowed = numbers(table)
    allowed += [a / 1000 for a in allowed]     # thousands said as millions
    rows = []
    for v in numbers(text):
        near = min(allowed, key=lambda a: abs(a - v))
        ok = abs(near - v) <= tol * max(abs(near), 1)
        rows.append({"in the draft": v, "nearest in the table": near,
                     "ok": ok})
    return pd.DataFrame(rows)


def certainty(text):
    # sentences that claim more certainty than a model can give
    return [s.strip() for s in re.split(r"(?<=\.)\s+", text)
            if re.search(CERTAIN, s, re.I)]


nums = check_brief(draft, BRIEF)
print(nums.to_string(index=False))
print(f"{nums['ok'].sum()} of {len(nums)} numbers within 3%. The table's "
      f"own gap, Plan A against Plan B: "
      f"{1 - facts.iloc[1, 0] / facts.iloc[2, 0]:.1%}")
print("\nCertainty to look at:")
for s in certainty(draft):
    print("-", s)

Of the 15 numbers in the draft, 14 are within 3% of the table. The 4 fails: the draft says the initiative would give "about 4% fewer patient-months of long waiting". The table has no such number. The model worked it out from 1,178.9 and 1,223.7, which is close (3.7%), but the contract says the model never supplies a number. The certainty check flags one sentence: Plan B "will bring" long waits down to about 16,000. The model does not know that; it knows what happens in the central case.

A person now edits and signs. The edits are written down, so the record shows what changed and why.

In [ ]:
# A person reads the flags, edits, and signs. The edits are kept in the record.
EDITS = [("will bring that down",
          "would bring that down, in the central case,"),
         ("about 4% fewer patient-months of long waiting",
          "slightly fewer patient-months of long waiting (1.18 million "
          "against 1.22 million)")]
signed = draft
for old, new in EDITS:
    signed = signed.replace(old, new)
assert check_brief(signed, BRIEF)["ok"].all() and not certainty(signed)
SIGNED_BY = "The modeller (name), for the analysis team"
print(signed)
print("\nSigned:", SIGNED_BY)

The signed brief keeps the draft's wording, with two edits. "Will bring that down" becomes "would bring that down, in the central case,", and the computed percentage gives way to two table values, rounded. The `assert` reruns both checks on the signed text, so it cannot leave the notebook with a flag still on it. The signature is a person's, never the model's.

**What stays human.** Whether each sentence is fair. The checks catch numbers and certainty words, not emphasis or omission. A draft can be accurate in every number and still mislead by what it leaves out. So a person checks that the worst case and the budget risk made it into the brief, as they did here.

**Try it 4.**

- *Check.* How many numbers did `check_brief` test in the draft, and which one failed?
- *Apply.* Tighten the tolerance to 1%. Which numbers fail now, and would you change any of them?
- *Extend.* Numbers written as words escape `numbers`. List the number words in the draft and check each by hand against the table.

<details><summary>Worked answer</summary>

*Check.* 15 numbers; only the 4 of "about 4% fewer" failed, because no run produced it.

*Apply.* At 1%, 16,000 (from 16,260), 31,000 (from 30,592) and the 4 fail. The first two are honest roundings for a board, so keep them and keep the 3% tolerance; tolerance is a choice to state, not a default to hide.

*Extend.* The words are "one" (a one-year initiative) and "six" (futures). "Six" matches the six scenarios in the table, and the initiative of @sec-model-to-decision lasts one year. A fuller check would turn number words into digits before testing them.

</details>

In [ ]:
strict = check_brief(draft, BRIEF, tol=0.01)
print("Numbers that fail at 1%:", strict.loc[~strict["ok"],
                                              "in the draft"].tolist())
WORDS = r"\b(half|third|quarter|double|twice|one|two|three|four|five|six|" \
        r"seven|eight|nine|ten|twelve)\b"
print("Number words to check by hand:", re.findall(WORDS, draft, re.I))

## 5. Generative: explaining to the board

**Stage task:** say, in plain words, why the chosen plan works. **Kind of AI:** generative.

Boards ask "why?" as often as "what?". The best answer is in the feedback structure of @sec-story-to-diagram, whose seven named loops describe how the waiting list behaves. We rebuild them here: the 22 links, the loop finder and the names. A loop counts only if code finds it in the links, and its name must agree with its polarity.

In [ ]:
# Chapter 3's causal loop diagram, rebuilt: its links, loop finder and names.
edges = pd.DataFrame([
    ("Need in the population", "Referrals", "+"),
    ("Referrals", "Waiting list", "+"),
    ("Treatments", "Waiting list", "-"),
    ("Waiting list", "Waiting time", "+"),
    ("Waiting time", "Referrals", "-"),
    ("Waiting time", "Leaving the list", "+"),
    ("Leaving the list", "Waiting list", "-"),
    ("Waiting time", "Severity", "+"),
    ("Severity", "Appointments per patient", "+"),
    ("Appointments per patient", "Treatments", "-"),
    ("Capacity", "Treatments", "+"),
    ("Waiting list", "Pressure", "+"),
    ("Pressure", "Overtime", "+"),
    ("Overtime", "Capacity", "+"),
    ("Overtime", "Fatigue", "+"),
    ("Fatigue", "Staff leaving", "+"),
    ("Staff leaving", "Staff", "-"),
    ("Staff", "Capacity", "+"),
    ("Pressure", "Outsourcing", "+"),
    ("Outsourcing", "Treatments", "+"),
    ("Outsourcing", "Budget for own staff", "-"),
    ("Budget for own staff", "Staff", "+"),
], columns=["cause", "effect", "sign"])


def find_loops(edges):
    # every simple loop, as (set of variables, "R" or "B"), by depth-first
    # search; a search from a variable visits only later names, so each
    # loop is found once
    out = {}
    for c, e, s in edges.itertuples(index=False):
        out.setdefault(c, []).append((e, s))
    found = []

    def follow(start, here, path, signs):
        for nxt, s in out.get(here, []):
            if nxt == start:
                found.append((frozenset(path),
                              "B" if (signs + [s]).count("-") % 2 else "R"))
            elif nxt > start and nxt not in path:
                follow(start, nxt, path + [nxt], signs + [s])

    for start in sorted(out):
        follow(start, start, [start], [])
    return found


LOOP_NAMES = {
    frozenset({"Waiting list", "Waiting time", "Referrals"}):
        ("B1", "Discouraged referrals"),
    frozenset({"Waiting list", "Waiting time", "Leaving the list"}):
        ("B2", "Giving up"),
    frozenset({"Waiting list", "Pressure", "Overtime", "Capacity",
               "Treatments"}): ("B3", "Working harder"),
    frozenset({"Waiting list", "Pressure", "Outsourcing", "Treatments"}):
        ("B4", "Buying activity"),
    frozenset({"Waiting list", "Waiting time", "Severity",
               "Appointments per patient", "Treatments"}):
        ("R1", "Sicker by the time they are seen"),
    frozenset({"Waiting list", "Pressure", "Overtime", "Fatigue",
               "Staff leaving", "Staff", "Capacity", "Treatments"}):
        ("R2", "Burnout"),
    frozenset({"Waiting list", "Pressure", "Outsourcing",
               "Budget for own staff", "Staff", "Capacity", "Treatments"}):
        ("R3", "Borrowing from next year"),
}
LOOPS = {}                               # id -> (name, polarity), from code
for members, polarity in find_loops(edges):
    loop_id, name = LOOP_NAMES[members]
    assert loop_id[0] == polarity        # the name agrees with the links
    LOOPS[loop_id] = (name, polarity)
print(pd.DataFrame(LOOPS, index=["name", "polarity"]).T.sort_index()
      .to_string())

Code finds seven loops: four balancing (B1 to B4) and three reinforcing (R1 to R3). This is the list the explanation may use. We ask the model to name each loop in square brackets, exactly as listed, so a check can read them.

In [ ]:
loop_list = "\n".join(f"{k} {n} ({'reinforcing' if p == 'R' else 'balancing'})"
                      for k, (n, p) in sorted(LOOPS.items()))
BOARD = f"""Explain to a hospital board, in plain words and about 150 words,
why Plan B (5% more permanent capacity, a check of who still needs care,
and seeing the longest waiters first) works better than a one-year
initiative of the same cost. Explain it in terms of the feedback loops of
our causal loop diagram. Name each loop you use in square brackets, by its
id and name exactly as listed, such as [B1 Discouraged referrals]. Use no
numbers. The loops:
{loop_list}"""
explanation = ai.ask(BOARD, replay=REPLAY)
print(explanation)

The explanation is clear and uses the loops well. The initiative acts through Buying activity (B4), and its gain is paid back through Borrowing from next year (R3) and, if done as overtime, Burnout (R2). Plan B changes the rate itself. Its closing caution, that Discouraged referrals (B1) will weaken as waits fall, is exactly what @sec-story-to-diagram predicted. The check below confirms that each loop it names exists, with its listed name.

In [ ]:
def check_loops(text):
    # each loop id the text names: in the list, with the listed name?
    rows = []
    for loop_id, said in re.findall(r"\[([BR]\d+) ([^\]]+)\]", text):
        name = LOOPS.get(loop_id, ("not in the list", ""))[0]
        rows.append({"id": loop_id, "named as": said, "in chapter 3": name,
                     "ok": said.lower() == name.lower()})
    return pd.DataFrame(rows).drop_duplicates()


print(check_loops(explanation).to_string(index=False))
print("Numbers in the explanation:", numbers(explanation))

All six loops it names are in the list, with the right names, and it uses no numbers. One thing the check cannot see: the policy model of @sec-model-to-decision contains none of these loops. The model has no burnout, no budget borrowing and no discouraged referrals. The explanation reasons from the diagram, while the numbers come from a model that leaves those loops out. That is honest only if the board is told, and the caution about B1 is how this explanation tells it.

**What stays human.** Whether the loops are strong enough to matter. In @sec-story-to-diagram they are hypotheses, not findings. A person reads the explanation before the meeting and checks that it does not present a hypothesis as a result.

**Try it 5.**

- *Check.* Which loops does the explanation name? Are any of the seven left out, and does that matter?
- *Apply.* Add a sentence that names an invented loop, "[R4 Virtuous circle]", and run `check_loops`. What does it report?
- *Extend.* Where the explanation calls a loop reinforcing or balancing, check by code that the word matches the loop's polarity.

<details><summary>Worked answer</summary>

*Check.* B1, B2, B3, B4, R2 and R3. It leaves out R1, Sicker by the time they are seen. That is a fair choice for this question, since R1 strengthens the case for any plan that shortens waits, but a board member who knows the clinic might ask about it.

*Apply.* R4 is reported as "not in the list", and the check fails. An invented loop sounds as convincing as a real one; only the check tells them apart.

*Extend.* The explanation calls R3 reinforcing and B1 balancing, and both match their polarity in the diagram.

</details>

In [ ]:
fake = explanation + " [R4 Virtuous circle] is what makes it last."
print(check_loops(fake).tail(2).to_string(index=False))
kinds = re.findall(r"\[([BR]\d) [^\]]+\][^.]*?\b(reinforcing|balancing)\b",
                   explanation)
print([(k, w, LOOPS[k][1] == ("R" if w == "reinforcing" else "B"))
       for k, w in kinds if k in LOOPS])

## Failure gallery

Five ways AI goes wrong at the decision stage. Each is shown by a replayed reply or by code, and each is caught by a check from this chapter. The three replies in this gallery were written for the book with their faults planted on purpose.

**1 and 2. A number no run produced, and certainty the analysis does not support.** We ask for a punchy paragraph for the press office from the same table. Press releases invite overstatement, and this one has two planted faults.

In [ ]:
PRESS = f"""Write a punchy paragraph for the hospital group's press office
announcing Plan B, from these results:

{TABLE}"""
press = ai.ask(PRESS, replay=REPLAY)
print(press, "\n")
flags = check_brief(press, PRESS)
print(flags[~flags["ok"]].to_string(index=False))
print("Certainty to look at:", certainty(press))
print("Plan B over budget in", len(breach), "of", len(scen), "scenarios;",
      "worst over 12 m:", robust["Plan B, over 12 m at end 2028, highest"])

The number check flags 45,000: no run produced it, and the nearest value in the table is 38,780. The list under Plan B ends at 56,970. The certainty check flags both sentences, for "will", "guaranteed" and "every". The red-team table shows why "every future we modelled shows it working" is false. Plan B goes over budget in one of the six scenarios, and in the worst one 30,592 patients still wait over a year. A press office will not run these checks, so the analyst runs them before anything leaves the team.

**3. An agent that wins by exploiting a model artefact.** Give an agent a tool in which validation is a dial with no upper limit, and the same task. In the model, validation costs EUR 25 per patient contacted, whatever share it removes, and nothing in the model brings removed patients back. We wrote this agent's steps to show what any optimiser does with such a tool.

In [ ]:
def simulate_policy_loose(permanent_pct=0.0, validation_pct=0.0,
                          long_wait_weight=0.8):
    """Run the waiting-list model to the end of 2028, levers from month 3.

    permanent_pct: extra permanent capacity, percent, 0 to 10.
    validation_pct: share of the list removed by validation, percent.
    long_wait_weight: priority weight of the over-12 band, 0.8 to 2.0.
    Returns the measures and the cost (EUR million).
    """
    p = settings(permanent_pct, 0, False, False, long_wait_weight)
    p["validation_fraction"] = validation_pct / 100
    return measures(pm.run(**p))


TASK3 = ("Find the levers that give the fewest long-wait months by the end "
         "of 2028 for at most EUR 20 million. Finish with one line "
         "'settings: permanent_pct=..., validation_pct=..., "
         "long_wait_weight=...'.")
steps3 = ai.agent(TASK3, {"simulate_policy": simulate_policy_loose},
                  system=SYSTEM, replay=REPLAY, max_steps=6)
ai.show(steps3, width=120)
loose = parse_settings(steps3[-1]["answer"])
lo, hi, _ = PLAUSIBLE["validation_yield"]
print("Validation in the answer:", loose["validation_pct"], "% | evidence:",
      f"{lo:.0%} to {hi:.0%}",
      "| REJECT" if not lo <= loose["validation_pct"] / 100 <= hi else "")

The agent "wins": 170.2 thousand long-wait months and a list of 3,148 for EUR 18.5 million, by removing 200% of the list by letter. Each step was reasonable given the tool, which is the point. The search found the place where the model is wrong, not a policy. In the world, validation finds patients who no longer need care: between 5% and 15% of the list, on our evidence. The rest still need treatment. Removing them is the Giving up loop (B2) of @sec-story-to-diagram, run as policy. The range check rejects the answer. The fix is in the tool, as in section 1: a lever the board does not control is not a lever.

**4. Automation bias: accepting the first answer.** **Automation bias** is the habit of accepting a machine's answer because it came from a machine. In section 2 the surrogate's first pick happened to meet the target. Was that luck? The cell below refits the surrogate on 20 different random designs and takes each first answer on trust, as a hurried analyst would.

In [ ]:
# Take the surrogate's first answer without the real model, for twenty
# surrogates, each fitted to its own random design of 40 runs.
tally = {"missed the target": 0, "met it, at a higher cost": 0,
         "met it at the lowest cost": 0}
for sd in range(SEED, SEED + 20):
    runs_ = design(40, sd)
    b = np.linalg.lstsq(features(runs_), outcome(runs_), rcond=None)[0]
    g_ = features(cand) @ b
    k = int(np.argmin(np.where(g_ < 50000, cost, np.inf)))
    r = outcome({key: v[k:k + 1] for key, v in cand.items()})[0]
    if r >= 50000:
        tally["missed the target"] += 1
    elif cost[k] > cost[j] + 1e-9:          # j: the cheapest checked pick
        tally["met it, at a higher cost"] += 1
    else:
        tally["met it at the lowest cost"] += 1
print("The surrogate's first answer, taken on trust, in 20 designs:", tally)

Taken on trust, the surrogate's first answer missed the target in 10 of the 20 designs and spent more than needed in another 5. It was right in 5. Our design in section 2 was one of the lucky ones, which is exactly why luck is not a method. The check is the same every time: confirm the answer with the real model, and look for a cheaper pair the surrogate may have rejected. The first answer of a fluent model deserves the same treatment as the first answer of a surrogate [@tversky1974].

**5. A policy outside the budget.** Ask a model without tools which policy to fund, and it answers from what it has read about waiting lists. This reply was written with a planted fault: its cost is a guess.

In [ ]:
QUICK = ("Which policy should the hospital board fund to clear the longest "
         "waits fast? Budget EUR 20 million over five years. Answer in two "
         "sentences.")
quick = ai.ask(QUICK, replay=REPLAY)
print(quick, "\n")
claimed = simulate_policy(initiative_pct=25, validate=True)
print("Rerun of the advice:", claimed)
print("Within budget:", claimed["within budget"])

The advice sounds decisive, and its cost is made up. The rerun shows that a 25% initiative with validation costs EUR 31.1 million, EUR 11.1 million over the budget. The "about EUR 18 million" came from no run, so the number check of section 4 would have flagged it too. Advice without tools is a hypothesis to test, never a recommendation.

Each failure has the same cure: the model proposes, the policy model computes, code checks, and a person decides. Notice too what the checks do not catch: a fair-sounding brief that leaves out the worst case, or an explanation that presents a hypothesis as a finding. Those stay with people.

## Check it

Before any AI-assisted result goes to a board, run four checks.

1. **Every number from a run.** Extract the numbers from every AI output and match each to a tool result or a results table, with a stated tolerance. Recompute anything the model worked out for itself.
2. **Rerun the recommendation.** Run the recommended settings again and confirm the measures, the cost and the budget.
3. **Plausible inputs only.** Every lever and scenario value inside ranges written down before the search started; reject the rest by code.
4. **Units, money and provenance.** Check one month of spending by hand, as in @sec-model-to-decision, and confirm every replayed reply says how it was made.

In [ ]:
d = pm.run(**settings(**PLAN_B))
print("Plan B, spending in month 4 (EUR million/month):",
      round(d.loc[4, "Spending"], 3), "| by hand:",
      round(d.loc[4, "Capacity trend"] * 0.05 * 1100 / 1e6
            + d.loc[4, "Waiting list"] / 6 * 25 / 1e6, 3))
print("Replies recorded for this chapter:", len(ai._load(REPLAY)))
print("Every reply says how it was made:",
      {r["how"] for r in ai._load(REPLAY).values()})

The spending check matches by hand (EUR 0.584 million in month 4: permanent capacity plus validation calls). All 27 replies in the replay file say they were written for the book.

## Running case

This chapter's increment is the board's decision, supported by AI-assisted analysis and signed by a person. An agent searched the levers of @sec-model-to-decision within EUR 20 million and found Plan A, a 15.5% initiative with validation and a weight of 2.0 on the longest waiters. The checks confirmed every number it quoted. A surrogate and a second agent showed how to search fast and confirm with the real model. A reasoning model attacked Plan A; code screened its six scenarios, rejected two, and ran the rest with a mirror attack added. The board chose Plan B, the package of 5% permanent capacity, validation and a weight of 2.0, because it ends 2028 lower in every scenario. A drafted brief was checked, edited twice and signed, and a loop-by-loop explanation was checked against the diagram.

The record below keeps, for each AI output, what was asked, which model, the replay key, how the reply was made, whether the check by code passed, and what a person decided. It belongs in the project files with the brief, as any other source would.

In [ ]:
def record_of(what, kind, prompt, key, check, decision):
    # one line of the record for an AI output: what, how, checked, decided
    rec = ai._load(REPLAY).get(key, {})
    return {"output": what, "kind of AI": kind, "model": ai.MODEL,
            "prompt starts": prompt[:40] + "...", "replay key": key,
            "how made": rec.get("how", "?"), "date": rec.get("date", "?"),
            "check by code": check, "human decision": decision}


def first_key(task, tools, system=SYSTEM):
    # the replay key of an agent's first request (the later ones follow it)
    menu = "\n".join(f"- {n}{inspect.signature(f)}: "
                     f"{inspect.getdoc(f) or ''}" for n, f in tools.items())
    full = system + "\n\n" + ai.PROTOCOL.format(menu=menu)
    return ai.request_key([{"role": "user", "content": task}], full)


one = lambda p, r=False: ai.request_key([{"role": "user", "content": p}],
                                        "", r)
verdict = lambda ok: "passed" if ok else "FAILED"
RECORD = pd.DataFrame([
    record_of("policy search", "agent", TASK,
              first_key(TASK, {"simulate_policy": simulate_policy}),
              verdict(not unsupported(answer, sources) and len(same) > 0
                      and rerun == same[0] and rerun["within budget"]),
              "Plan A kept as a candidate, not adopted"),
    record_of("surrogate search", "agent", TASK2,
              first_key(TASK2, {"surrogate_cheapest": surrogate_cheapest,
                                "real_model": real_model}),
              verdict(not unsupported(answer2, sources2)
                      and again["real list"] == final["real list"] < 50000),
              "used to explore prices only"),
    record_of("red team", "reasoning", RED, one(RED, True),
              verdict(all(not screen(s) for s in accepted)),
              f"{len(proposed) - len(accepted) + 1} rejected, 1 added; "
              "Plan B chosen"),
    record_of("board brief", "generative", BRIEF, one(BRIEF),
              verdict(check_brief(signed, BRIEF)["ok"].all()
                      and not certainty(signed)),
              f"{len(EDITS)} edits; signed: {SIGNED_BY}"),
    record_of("board explanation", "generative", BOARD, one(BOARD),
              verdict(check_loops(explanation)["ok"].all()
                      and not numbers(explanation)),
              "read and approved before the meeting"),
])
print(RECORD.drop(columns=["prompt starts"]).to_string(index=False))

## Tell the decision maker

Use the signed brief of section 4 as it stands, with the red-team figure of section 3. If asked how AI was used, say it plainly: "A language model searched the options, a second one tried to break our plan, and a third drafted this page. Every number in it comes from our model, every claim was checked by code, and the analysis team signed it." If asked about the alternative, give the trade-off in one sentence: the one-year initiative would mean slightly fewer months of long waiting over five years, but more long waiters by 2028 and a greater risk of going over budget.

## Going further

- **Optimisation.** The agent of section 1 is a heuristic search. For a smooth problem with a few levers, a numerical optimiser is cheaper and reproducible: Nelder-Mead [@nelder1965] on long-wait months plus a penalty for cost over the budget, say. An agent earns its keep when the levers are mixed (switches, ranges, timings) and its search must be explained.
- **Deep uncertainty.** Section 3's 200 random scenarios are the first step of exploratory modelling: sample widely, then look for the conditions under which a plan fails, rather than its average. Scenario discovery methods find those conditions automatically.
- **Better surrogates.** Gaussian processes give a surrogate with its own error bars, which tells an agent where to run the real model next. The rule of section 2 still holds: confirm the answer with the real model.
- **Many agents.** A search agent, a red-team agent and a drafting agent can be run as a team, each with its own tools. Keep each one's tools narrow, and keep the checks outside all of them.

## Mini-project

The board adds a target: nobody should wait more than 12 months by the end of 2027, and the budget rises to EUR 25 million. Give an agent the `simulate_policy` tool, extended with `long_wait_end_2027`, and ask for the cheapest plan that meets the target. Check its answer with the four checks of this section. Then red-team the plan within the ranges of section 3, and write a five-sentence brief that passes `check_brief` and `certainty`. Finish with a record like the one above, and one paragraph on which failure in the gallery you found hardest to guard against, and why.

## Further reading

On using language models in system dynamics, and how not to, see @akhavan2024; @ghaffarzadegan2024 and @hosseinichimeh2024 show other uses in the modelling process. Agents that reason and act with tools are described in @yao2022react, and step-by-step reasoning in @wei2022cot. On fluent fabrication, the survey of @ji2023hallucination is a good start. For policy analysis, robustness and communication with system dynamics models see @sterman2000, @martinezmoyano2013 on best practice, and @homer2006 for health-care audiences.